# 03. External Validation (KNHANES 2016–2024)

Evaluate the calibrated voting ensemble on the Korea National Health and Nutrition Examination Survey (KNHANES) external cohort.

Reproduces:
- Table 2 — external validation performance (AUROC, sensitivity, specificity, PPV, NPV, Brier)
- Figure 1 — ROC curves (4 targets)
- Figure 2 — Subgroup forest plot (age × sex × survey period)
- Figure 3 — Calibration plots after isotonic recalibration
- eTables 7–11 — multiple-threshold and sensitivity analyses

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import joblib, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

from data_utils import BASE_FEATURES
from eval_utils import bootstrap_metrics, evaluate_binary

KN_PATH = '../data/processed/knhanes_external.csv'
MODELS  = '../models'

In [ ]:
kn = pd.read_csv(KN_PATH, low_memory=False)

TARGETS = {
    'uric70':  ('HE_glu',   126, 'FPG ≥ 126 mg/dL'),
    'uric100':  ('HE_glu',   100, 'FPG ≥ 100 mg/dL'),
    'uric90': ('HE_HbA1c', 6.5, 'HbA1c ≥ 6.5%'),
    'uric_internal': ('HE_HbA1c', 5.7, 'HbA1c ≥ 5.7%'),
}

results = {}
for name, (col, thr, label) in TARGETS.items():
    sub = kn[BASE_FEATURES + [col]].dropna()
    y = (sub[col] >= thr).astype(int).values
    model = joblib.load(f'{MODELS}/{name}_calibrated_voting.joblib')
    yp = model.predict_proba(sub[BASE_FEATURES])[:, 1]
    m  = bootstrap_metrics(y, yp, n_bootstrap=1000)
    brier = brier_score_loss(y, yp)
    results[name] = {'y': y, 'yp': yp, 'metrics': m, 'brier': brier, 'label': label}
    print(f'{label:30s}  AUROC={m["AUC"]:.3f} ({m["AUC_CI"][0]:.3f}–{m["AUC_CI"][1]:.3f})  Brier={brier:.3f}')

## Figure 1 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(r['y'], r['yp'])
    ls = '-' if 'uric70' in name or 'uric90' in name else '--'
    ax.plot(fpr, tpr, lw=2.5, ls=ls, label=f"{r['label']} (AUROC = {r['metrics']['AUC']:.3f})")
ax.plot([0,1],[0,1], color='#888', ls=':', lw=1.2)
ax.set_xlabel('1 − Specificity'); ax.set_ylabel('Sensitivity')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

## Figure 3 — Calibration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, (name, r) in zip(axes.flat, results.items()):
    pt, pp = calibration_curve(r['y'], r['yp'], n_bins=10, strategy='quantile')
    ax.plot(pp, pt, 'o-', lw=2, markersize=7, label='Model')
    ax.plot([0, max(pp.max(), pt.max())], [0, max(pp.max(), pt.max())], 'k--', lw=1.2, label='Perfect')
    ax.set_title(f"{r['label']}  (Brier={r['brier']:.3f})")
    ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed proportion')
    ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()